# Cross-Modal Satellite Image Retrieval (BigEarthNet-14K)
Sanitized end-to-end pipeline for training, evaluating, and visualizing cross-modal and modality-agnostic satellite image retrieval across Sentinel-1 (SAR) and Sentinel-2 (Optical/Multispectral) GeoTIFF rasters.

In [ ]:
# ==============================================================================
# 1. ENVIRONMENT SETUP & DEPENDENCIES
# ==============================================================================
from google.colab import drive
drive.mount('/content/drive')

!pip install -q rasterio faiss-cpu

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Free VRAM:", round(torch.cuda.mem_get_info()[0] / 1024**3, 2), "GB")

In [ ]:
# ==============================================================================
# 2. DATASET DOWNLOAD & EXTRACTION
# ==============================================================================
import os
import zipfile
from pathlib import Path

!kaggle datasets download -d narendraaironi/bigearthnet-14k -p /content/datasets

archive = "/content/datasets/bigearthnet-14k.zip"
EXTRACT_ROOT = "/content/datasets/BEN14K"
DATA_ROOT = "/content/datasets/BEN14K/BEN_14k"

os.makedirs(EXTRACT_ROOT, exist_ok=True)
with zipfile.ZipFile(archive, "r") as z:
    z.extractall(EXTRACT_ROOT)

root = Path(DATA_ROOT)
expected = {"train": 7180, "validation": 3255, "test": 3248}

for modality in ["BigEarthNet-S1", "BigEarthNet-S2"]:
    print(f"\n{modality}")
    for split, expected_count in expected.items():
        folder = root / modality / split
        count = len(list(folder.glob("*.tif")))
        status = "OK" if count == expected_count else "MISMATCH"
        print(f"  {split}: {count:,} / {expected_count:,} [{status}]")

print("\nMetadata exists:", (root / "metadata.parquet").exists())

In [ ]:
# ==============================================================================
# 3. DIAGNOSTIC RASTER INSPECTION & PAIRED MANIFEST CREATION
# ==============================================================================
import os
import glob
import numpy as np
import pandas as pd
import rasterio

DATA_ROOT = "/content/datasets/BEN14K/BEN_14k"
metadata_path = os.path.join(DATA_ROOT, "metadata.parquet")

assert os.path.exists(metadata_path), f"Cannot find metadata at {metadata_path}!"
df = pd.read_parquet(metadata_path)

s1_files = glob.glob(os.path.join(DATA_ROOT, "BigEarthNet-S1", "*", "*.tif"))
s2_files = glob.glob(os.path.join(DATA_ROOT, "BigEarthNet-S2", "*", "*.tif"))

s1_disk_map = {os.path.splitext(os.path.basename(f))[0]: f for f in s1_files}
s2_disk_map = {os.path.splitext(os.path.basename(f))[0]: f for f in s2_files}

df['s1_path'] = df['s1_name'].map(s1_disk_map)
df['s2_path'] = df['patch_id'].map(s2_disk_map)

paired_df = df.dropna(subset=['s1_path', 's2_path']).copy()
print(f"Verified matching S1-S2 pairs: {len(paired_df)}")
print(paired_df['split'].value_counts())

manifest_path = os.path.join(DATA_ROOT, "verified_paired_manifest.csv")
paired_df.to_csv(manifest_path, index=False)
print(f"Saved verified paired manifest to: {manifest_path}\n")

def inspect_raster(path, modality_tag):
    with rasterio.open(path) as src:
        arr = src.read()
        print(f"[{modality_tag}] File: {os.path.basename(path)}")
        print(f"  Shape (C, H, W): {arr.shape} | Dtype: {arr.dtype} | CRS: {src.crs} | NoData: {src.nodata}")
        print(f"  Min: {np.min(arr, axis=(1, 2))}")
        print(f"  Max: {np.max(arr, axis=(1, 2))}\n")
    return arr

sample_row = paired_df[paired_df['split'] == 'train'].iloc[0]
_ = inspect_raster(sample_row['s1_path'], "Sentinel-1 (SAR)")
_ = inspect_raster(sample_row['s2_path'], "Sentinel-2 (Optical/MS)")

In [ ]:
# ==============================================================================
# 4. MODEL ARCHITECTURE, LOSS & END-TO-END TRAINING PIPELINE
# ==============================================================================
import os
import time
import copy
import random
import shutil
import re
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import rasterio
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.models as models
import faiss

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

DATA_ROOT = "/content/datasets/BEN14K/BEN_14k"
MANIFEST_PATH = os.path.join(DATA_ROOT, "verified_paired_manifest.csv")
BACKUP_DIR = "/content/drive/MyDrive/Satellite_Retrieval_Weights"
os.makedirs(BACKUP_DIR, exist_ok=True)
CHECKPOINT_NAME = "momentum_trimodal_best_model_80plus.pt"
DRIVE_CKPT = os.path.join(BACKUP_DIR, CHECKPOINT_NAME)

def parse_labels(raw_labels):
    """Robustly parses string, list, or numpy array of BigEarthNet multi-labels."""
    if isinstance(raw_labels, (list, np.ndarray)):
        return set(str(x).strip(" '\"") for x in raw_labels if len(str(x).strip(" '\"")) > 0)
    parts = re.findall(r"['\"]([^'\"]+)['\"]", str(raw_labels))
    if not parts:
        parts = str(raw_labels).strip("[]").replace('|', ',').split(',')
    return set(p.strip(" '\"") for p in parts if len(p.strip(" '\"")) > 0)

class BEN14kTiffDataset(torch.utils.data.Dataset):
    def __init__(self, manifest_path, split='train'):
        self.df = pd.read_csv(manifest_path)
        self.df = self.df[self.df['split'] == split].reset_index(drop=True)
        self.split = split
        print(f"[{split.upper()}] Loaded {len(self.df)} paired records.")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        with rasterio.open(row['s1_path']) as src_s1:
            s1_raw = src_s1.read().astype(np.float32)
        s1_norm = (np.clip(s1_raw, -35.0, 0.0) + 35.0) / 35.0

        with rasterio.open(row['s2_path']) as src_s2:
            s2_raw = src_s2.read().astype(np.float32)
        s2_norm = np.clip(s2_raw / 10000.0, 0.0, 1.0)

        return {
            's1': torch.from_numpy(s1_norm).float(),
            's2': torch.from_numpy(s2_norm).float(),
            'labels': str(row['labels']),
            'patch_id': row['patch_id']
        }

def apply_coregistered_augmentations(s1, s2):
    B = s1.shape[0]
    dev = s1.device
    flip_h = (torch.rand(B, 1, 1, 1, device=dev) > 0.5)
    flip_w = (torch.rand(B, 1, 1, 1, device=dev) > 0.5)
    s1 = torch.where(flip_h, torch.flip(s1, [2]), s1)
    s2 = torch.where(flip_h, torch.flip(s2, [2]), s2)
    s1 = torch.where(flip_w, torch.flip(s1, [3]), s1)
    s2 = torch.where(flip_w, torch.flip(s2, [3]), s2)
    rotations = torch.randint(0, 4, (B,), device=dev)
    for k in range(1, 4):
        mask = (rotations == k).view(-1, 1, 1, 1)
        s1 = torch.where(mask, torch.rot90(s1, k=k, dims=[2, 3]), s1)
        s2 = torch.where(mask, torch.rot90(s2, k=k, dims=[2, 3]), s2)
    return s1, s2

class GeM(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps
    def forward(self, x):
        return F.avg_pool2d(x.clamp(min=self.eps).pow(self.p), (x.size(-2), x.size(-1))).pow(1.0 / self.p)

class SensorBackbone(nn.Module):
    def __init__(self, in_channels, embed_dim=256):
        super().__init__()
        convnext = models.convnext_tiny(weights="IMAGENET1K_V1")
        old_weight = convnext.features[0][0].weight.data
        convnext.features[0][0] = nn.Conv2d(in_channels, 96, kernel_size=4, stride=4, bias=True)
        if in_channels == 2:
            convnext.features[0][0].weight.data = old_weight.mean(dim=1, keepdim=True).repeat(1, 2, 1, 1)
        elif in_channels == 10:
            convnext.features[0][0].weight.data = old_weight.repeat(1, 4, 1, 1)[:, :10, :, :] / 3.33

        self.features = convnext.features
        self.gem = GeM()
        self.projector = nn.Sequential(
            nn.Linear(768, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(512, embed_dim)
        )

    def forward(self, x):
        feat = self.features(x)
        pooled = self.gem(feat).flatten(1)
        return F.normalize(self.projector(pooled), p=2, dim=-1)

class CrossModalSAROpticalNet(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()
        self.sar_online = SensorBackbone(in_channels=2, embed_dim=embed_dim)
        self.opt_online = SensorBackbone(in_channels=10, embed_dim=embed_dim)
        self.sar_target = copy.deepcopy(self.sar_online)
        self.opt_target = copy.deepcopy(self.opt_online)
        for p in self.sar_target.parameters(): p.requires_grad = False
        for p in self.opt_target.parameters(): p.requires_grad = False

    @torch.no_grad()
    def update_targets(self, m=0.996):
        for o, t in [(self.sar_online, self.sar_target), (self.opt_online, self.opt_target)]:
            for q, k in zip(o.parameters(), t.parameters()):
                k.data = k.data * m + q.data * (1.0 - m)

    def forward(self, s1_aug, s2_aug, s1_orig, s2_orig):
        z_s1 = self.sar_online(s1_aug)
        z_s2 = self.opt_online(s2_aug)
        with torch.no_grad():
            t_s1 = self.sar_target(s1_orig)
            t_s2 = self.opt_target(s2_orig)
        return (z_s1, z_s2), (t_s1, t_s2)

def get_iou_matrix(labels, device):
    B = len(labels)
    iou = torch.zeros((B, B), device=device)
    split_sets = [parse_labels(l) for l in labels]
    for i in range(B):
        for j in range(B):
            inter = len(split_sets[i] & split_sets[j])
            union = len(split_sets[i] | split_sets[j])
            iou[i, j] = inter / (union + 1e-8)
    return iou

def symmetric_semantic_infonce(va, vb, iou_matrix, temp=0.1, threshold=0.40):
    logits = torch.mm(va, vb.t()) / temp
    positives = logits.diag()
    B = logits.size(0)
    ignore_mask = (iou_matrix > threshold) & (~torch.eye(B, device=logits.device, dtype=torch.bool))
    logits_neg = logits.clone()
    logits_neg[ignore_mask] = -1e9
    l_ab = -positives + torch.logsumexp(logits_neg, dim=1)
    l_ba = -positives + torch.logsumexp(logits_neg.t(), dim=1)
    return (l_ab + l_ba).mean() / 2.0

def compute_loss(online_vecs, target_vecs, labels):
    z_s1, z_s2 = online_vecs
    t_s1, t_s2 = target_vecs
    iou = get_iou_matrix(labels, z_s1.device)
    l_cross = 0.5 * (symmetric_semantic_infonce(z_s1, t_s2, iou) + symmetric_semantic_infonce(z_s2, t_s1, iou))
    l_s1 = symmetric_semantic_infonce(z_s1, t_s1, iou)
    l_s2 = symmetric_semantic_infonce(z_s2, t_s2, iou)
    return l_cross + 0.25 * (l_s1 + l_s2)

def evaluate_benchmark(model, dataloader, device):
    model.eval()
    all_s1, all_s2, all_labels = [], [], []
    with torch.no_grad():
        for b in dataloader:
            s1 = b['s1'].to(device)
            s2 = b['s2'].to(device)
            all_s1.append(model.sar_target(s1).cpu())
            all_s2.append(model.opt_target(s2).cpu())
            all_labels.extend([parse_labels(s) for s in b['labels']])

    v_s1 = torch.cat(all_s1).numpy().astype(np.float32)
    v_s2 = torch.cat(all_s2).numpy().astype(np.float32)
    N = len(all_labels)

    index_s2 = faiss.IndexFlatIP(256)
    index_s2.add(v_s2)
    index_s1 = faiss.IndexFlatIP(256)
    index_s1.add(v_s1)

    t0 = time.perf_counter()
    _, hits_s1_to_s2 = index_s2.search(v_s1, 10)
    _, hits_s2_to_s1 = index_s1.search(v_s2, 10)
    latency_ms = ((time.perf_counter() - t0) / (2 * N)) * 1000.0

    rec1_s1_s2 = np.mean([1.0 if i in hits_s1_to_s2[i][:1] else 0.0 for i in range(N)])
    rec5_s1_s2 = np.mean([1.0 if i in hits_s1_to_s2[i][:5] else 0.0 for i in range(N)])
    rec10_s1_s2 = np.mean([1.0 if i in hits_s1_to_s2[i][:10] else 0.0 for i in range(N)])

    f1_5_cross, f1_10_cross = [], []
    for i in range(N):
        q_lbl = all_labels[i]
        valid_hits = [h for h in hits_s1_to_s2[i] if h != i]
        top5, top10 = valid_hits[:5], valid_hits[:10]
        if top5:
            f1_5_cross.append(np.mean([2 * len(q_lbl & all_labels[h]) / (len(q_lbl) + len(all_labels[h])) for h in top5]))
        if top10:
            f1_10_cross.append(np.mean([2 * len(q_lbl & all_labels[h]) / (len(q_lbl) + len(all_labels[h])) for h in top10]))

    model.train()
    return {
        'Geo_R@1': rec1_s1_s2,
        'Geo_R@5': rec5_s1_s2,
        'Geo_R@10': rec10_s1_s2,
        'Semantic_F1@5': np.mean(f1_5_cross),
        'Semantic_F1@10': np.mean(f1_10_cross),
        'Latency_ms': latency_ms
    }

BATCH_SIZE = 32
EPOCHS = 30
LR = 0.0003

train_dataset = BEN14kTiffDataset(MANIFEST_PATH, split='train')
val_dataset = BEN14kTiffDataset(MANIFEST_PATH, split='validation')

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

model = CrossModalSAROpticalNet(embed_dim=256).to(device)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None

best_score = 0.0
local_ckpt = CHECKPOINT_NAME

for epoch in range(EPOCHS):
    model.train()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for b in pbar:
        s1_orig, s2_orig = b['s1'].to(device), b['s2'].to(device)
        s1_aug, s2_aug = apply_coregistered_augmentations(s1_orig, s2_orig)
        optimizer.zero_grad()
        if scaler is not None:
            with torch.amp.autocast('cuda', dtype=torch.float16):
                online_v, target_v = model(s1_aug, s2_aug, s1_orig, s2_orig)
                loss = compute_loss(online_v, target_v, b['labels'])
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            online_v, target_v = model(s1_aug, s2_aug, s1_orig, s2_orig)
            loss = compute_loss(online_v, target_v, b['labels'])
            loss.backward()
            optimizer.step()
        model.update_targets(m=0.996)
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})
    scheduler.step()

    if (epoch + 1) % 3 == 0 or epoch == EPOCHS - 1:
        metrics = evaluate_benchmark(model, val_loader, device)
        print(f"\n[VAL Epoch {epoch+1}] R@1: {metrics['Geo_R@1']:.4f} | R@5: {metrics['Geo_R@5']:.4f} | R@10: {metrics['Geo_R@10']:.4f} | F1@5: {metrics['Semantic_F1@5']:.4f}")
        combined = metrics['Geo_R@5'] + metrics['Semantic_F1@5']
        if combined >= best_score:
            best_score = combined
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'metrics': metrics
            }, local_ckpt)
            shutil.copy(local_ckpt, DRIVE_CKPT)
            print(f"  --> Synced checkpoint to Drive: {DRIVE_CKPT}")

# Force OS buffer synchronization to ensure weights are completely flushed to Google Drive
if hasattr(os, 'sync'):
    os.sync()

assert os.path.exists(DRIVE_CKPT), f"CRITICAL: Checkpoint {DRIVE_CKPT} not found on Drive!"
drive_size_mb = os.path.getsize(DRIVE_CKPT) / (1024 * 1024)
print(f"\n[VERIFIED] Checkpoint successfully persisted to Google Drive:")
print(f"  Path: {DRIVE_CKPT}")
print(f"  Size: {drive_size_mb:.2f} MB")

In [ ]:
# ==============================================================================
# 5. UNIFIED MODALITY-AGNOSTIC QUANTITATIVE EVALUATOR
# ==============================================================================
import time
import numpy as np
import torch
import faiss

def evaluate_unified_modality_agnostic(model, dataloader, device):
    model.eval()
    sar_vecs, opt_vecs, all_labels = [], [], []

    print("Extracting test embeddings across both sensor modalities...")
    with torch.no_grad():
        for b in dataloader:
            s1 = b['s1'].to(device)
            s2 = b['s2'].to(device)
            sar_vecs.append(model.sar_target(s1).cpu())
            opt_vecs.append(model.opt_target(s2).cpu())
            for lbl in b['labels']:
                all_labels.append(parse_labels(lbl))

    v_sar = torch.cat(sar_vecs).numpy().astype(np.float32)
    v_opt = torch.cat(opt_vecs).numpy().astype(np.float32)
    N = len(all_labels)

    # Latent Centering Calibration
    v_sar_cal = v_sar - np.mean(v_sar, axis=0, keepdims=True)
    v_opt_cal = v_opt - np.mean(v_opt, axis=0, keepdims=True)
    faiss.normalize_L2(v_sar_cal)
    faiss.normalize_L2(v_opt_cal)

    # Unified Gallery (2 * N items)
    unified_embeddings = np.vstack([v_sar_cal, v_opt_cal])
    gallery_pids = np.array(list(range(N)) * 2)
    gallery_mods = np.array(['SAR'] * N + ['Optical'] * N)
    gallery_labels = all_labels * 2
    total_items = len(unified_embeddings)

    unified_index = faiss.IndexFlatIP(256)
    unified_index.add(unified_embeddings)
    print(f"Unified Gallery built: {unified_index.ntotal} items indexed.")

    # Search
    t0 = time.perf_counter()
    _, raw_top_hits = unified_index.search(unified_embeddings, 25)
    total_time_ms = (time.perf_counter() - t0) * 1000.0
    avg_latency_ms = total_time_ms / total_items

    geo_r1, geo_r5, geo_r10 = [], [], []
    sem_f1_5, sem_f1_10 = [], []
    retrieved_sensor_distribution = {'SAR': 0, 'Optical': 0}

    breakdown = {
        'SAR': {'geo_r1': [], 'geo_r5': [], 'geo_r10': [], 'f1_5': [], 'f1_10': []},
        'Optical': {'geo_r1': [], 'geo_r5': [], 'geo_r10': [], 'f1_5': [], 'f1_10': []}
    }

    for q_idx in range(total_items):
        q_pid = gallery_pids[q_idx]
        q_mod = gallery_mods[q_idx]
        q_lbl = gallery_labels[q_idx]

        hits = [
            h for h in raw_top_hits[q_idx] 
            if not (gallery_pids[h] == q_pid and gallery_mods[h] == q_mod)
        ]

        for h in hits[:5]:
            retrieved_sensor_distribution[gallery_mods[h]] += 1

        twin_found_flags = [gallery_pids[h] == q_pid for h in hits]
        r1 = 1.0 if any(twin_found_flags[:1]) else 0.0
        r5 = 1.0 if any(twin_found_flags[:5]) else 0.0
        r10 = 1.0 if any(twin_found_flags[:10]) else 0.0

        geo_r1.append(r1)
        geo_r5.append(r5)
        geo_r10.append(r10)
        breakdown[q_mod]['geo_r1'].append(r1)
        breakdown[q_mod]['geo_r5'].append(r5)
        breakdown[q_mod]['geo_r10'].append(r10)

        sem_hits = [h for h in hits if gallery_pids[h] != q_pid]
        top5_sem = sem_hits[:5]
        top10_sem = sem_hits[:10]

        if top5_sem:
            f1_5_vals = [
                2 * len(q_lbl & gallery_labels[h]) / (len(q_lbl) + len(gallery_labels[h]))
                for h in top5_sem if (len(q_lbl) + len(gallery_labels[h])) > 0
            ]
            score_5 = np.mean(f1_5_vals) if f1_5_vals else 0.0
            sem_f1_5.append(score_5)
            breakdown[q_mod]['f1_5'].append(score_5)

        if top10_sem:
            f1_10_vals = [
                2 * len(q_lbl & gallery_labels[h]) / (len(q_lbl) + len(gallery_labels[h]))
                for h in top10_sem if (len(q_lbl) + len(gallery_labels[h])) > 0
            ]
            score_10 = np.mean(f1_10_vals) if f1_10_vals else 0.0
            sem_f1_10.append(score_10)
            breakdown[q_mod]['f1_10'].append(score_10)

    report = {
        'total_queries': total_items,
        'gallery_size': total_items,
        'latency_ms': avg_latency_ms,
        'overall': {
            'Geo_R@1': np.mean(geo_r1),
            'Geo_R@5': np.mean(geo_r5),
            'Geo_R@10': np.mean(geo_r10),
            'Semantic_F1@5': np.mean(sem_f1_5),
            'Semantic_F1@10': np.mean(sem_f1_10)
        },
        'sar_queries': {
            'Geo_R@1': np.mean(breakdown['SAR']['geo_r1']),
            'Geo_R@5': np.mean(breakdown['SAR']['geo_r5']),
            'Geo_R@10': np.mean(breakdown['SAR']['geo_r10']),
            'Semantic_F1@5': np.mean(breakdown['SAR']['f1_5']),
            'Semantic_F1@10': np.mean(breakdown['SAR']['f1_10'])
        },
        'optical_queries': {
            'Geo_R@1': np.mean(breakdown['Optical']['geo_r1']),
            'Geo_R@5': np.mean(breakdown['Optical']['geo_r5']),
            'Geo_R@10': np.mean(breakdown['Optical']['geo_r10']),
            'Semantic_F1@5': np.mean(breakdown['Optical']['f1_5']),
            'Semantic_F1@10': np.mean(breakdown['Optical']['f1_10'])
        },
        'top5_sensor_distribution': {
            'SAR_ratio': retrieved_sensor_distribution['SAR'] / (total_items * 5),
            'Optical_ratio': retrieved_sensor_distribution['Optical'] / (total_items * 5)
        }
    }
    # Return report along with raw test vectors for Section 6
    return report, v_sar, v_opt, all_labels

# ==============================================================================
# EXECUTE BENCHMARK ON UNSEEN TEST SPLIT
# ==============================================================================
test_dataset = BEN14kTiffDataset(MANIFEST_PATH, split='test')
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)

eval_model = CrossModalSAROpticalNet(embed_dim=256).to(device)
checkpoint = torch.load(DRIVE_CKPT, map_location=device, weights_only=False)
eval_model.load_state_dict(checkpoint['model_state_dict'])

metrics, v_sar, v_opt, all_labels = evaluate_unified_modality_agnostic(eval_model, test_loader, device)
N = len(all_labels)

print("\n" + "=" * 70)
print(f"UNIFIED MODALITY-AGNOSTIC BENCHMARK REPORT ({metrics['total_queries']} TOTAL QUERIES)")
print("=" * 70)
print("1. OVERALL POOLED METRICS (Across Both Modalities):")
print(f"   --> Geographic Counterpart R@1:   {metrics['overall']['Geo_R@1']*100:.2f}%")
print(f"   --> Geographic Counterpart R@5:   {metrics['overall']['Geo_R@5']*100:.2f}%")
print(f"   --> Geographic Counterpart R@10:  {metrics['overall']['Geo_R@10']*100:.2f}%")
print(f"   --> Semantic Multi-label F1@5:    {metrics['overall']['Semantic_F1@5']:.4f}")
print(f"   --> Semantic Multi-label F1@10:   {metrics['overall']['Semantic_F1@10']:.4f}")
print("=" * 70)

In [ ]:
# ==============================================================================
# 6. MODALITY-AGNOSTIC VISUAL RETRIEVAL (STRICT LEAK-FREE INFERENCE)
# ==============================================================================
import time
import random
import textwrap
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
import faiss

eval_model.eval()

# ------------------------------------------------------------------------------
# A. CALIBRATION CENTROIDS (Frozen from Validation Set, Never Test Set)
# ------------------------------------------------------------------------------
# Extract validation means to avoid test-set transductive leakage
val_sar_list, val_opt_list = [], []
with torch.no_grad():
    for b in val_loader:
        s1 = b['s1'].to(device)
        s2 = b['s2'].to(device)
        val_sar_list.append(eval_model.sar_target(s1).cpu())
        val_opt_list.append(eval_model.opt_target(s2).cpu())

mu_sar = torch.cat(val_sar_list).mean(dim=0, keepdim=True).numpy().astype(np.float32)
mu_opt = torch.cat(val_opt_list).mean(dim=0, keepdim=True).numpy().astype(np.float32)

# ------------------------------------------------------------------------------
# B. BUILD FROZEN UNIFIED GALLERY (TEST SPLIT)
# ------------------------------------------------------------------------------
# v_sar and v_opt were extracted from test_loader in Section 5
v_sar_cal = v_sar - mu_sar
v_opt_cal = v_opt - mu_opt
faiss.normalize_L2(v_sar_cal)
faiss.normalize_L2(v_opt_cal)

unified_embeddings = np.vstack([v_sar_cal, v_opt_cal]) # (6496, 256)
gallery_labels = all_labels * 2
gallery_mods = np.array(['SAR'] * N + ['Optical'] * N)
gallery_pids = np.array(list(range(N)) * 2)

agnostic_index = faiss.IndexFlatIP(256)
agnostic_index.add(unified_embeddings)

# ------------------------------------------------------------------------------
# C. VISUALIZATION HELPERS
# ------------------------------------------------------------------------------
def get_display_img(dataset, idx, modality):
    item = dataset[idx]
    if modality == 'SAR':
        sar = item['s1'][0].cpu().numpy()
        p2, p98 = np.percentile(sar, (2, 98))
        return np.clip((sar - p2) / (p98 - p2 + 1e-6), 0.0, 1.0), 'gray'
    else:
        # Optical RGB preview (Bands 2, 1, 0)
        rgb = item['s2'][[2, 1, 0], :, :].permute(1, 2, 0).cpu().numpy()
        p2, p98 = np.percentile(rgb, (2, 98))
        return np.clip((rgb - p2) / (p98 - p2 + 1e-6), 0.0, 1.0), None

def format_title_text(label_set, max_items=2, width=22):
    items = list(label_set)[:max_items]
    if len(label_set) > max_items:
        items.append(f"(+{len(label_set) - max_items} more)")
    text = "\n".join(items)
    return "\n".join(textwrap.wrap(text, width=width)) if len(text) > width else text

# ------------------------------------------------------------------------------
# D. INFERENCE WITH END-TO-END TIMING
# ------------------------------------------------------------------------------
def run_modality_agnostic_search(num_tests=4, top_k=5):
    for test_idx in range(num_tests):
        q_pid = random.randint(0, N - 1)
        q_mod = random.choice(['SAR', 'Optical'])
        q_lbl = all_labels[q_pid]
        query_item = test_dataset[q_pid]

        # 1. Feature Extraction Timing (Encoder forward pass)
        # 1. Feature Extraction Timing with Explicit CUDA Stream Synchronization
        if device.type == 'cuda':
            torch.cuda.synchronize()
        t_enc_start = time.perf_counter()

        with torch.no_grad():
            if q_mod == 'SAR':
                raw_tensor = query_item['s1'].unsqueeze(0).to(device)
                z = eval_model.sar_target(raw_tensor)
                if device.type == 'cuda':
                    torch.cuda.synchronize()  # Block until the ConvNeXt kernels finish on GPU
                z_np = z.cpu().numpy().astype(np.float32)
                z_cal = z_np - mu_sar
            else:
                raw_tensor = query_item['s2'].unsqueeze(0).to(device)
                z = eval_model.opt_target(raw_tensor)
                if device.type == 'cuda':
                    torch.cuda.synchronize()  # Block until the ConvNeXt kernels finish on GPU
                z_np = z.cpu().numpy().astype(np.float32)
                z_cal = z_np - mu_opt

            faiss.normalize_L2(z_cal)

        enc_latency_ms = (time.perf_counter() - t_enc_start) * 1000.0

        # 2. FAISS Nearest-Neighbor Index Search Timing
        t_faiss_start = time.perf_counter()
        distances, raw_hits = agnostic_index.search(z_cal, top_k + 5)
        faiss_latency_ms = (time.perf_counter() - t_faiss_start) * 1000.0
        total_latency_ms = enc_latency_ms + faiss_latency_ms

        # 3. Filter Self-Match (same patch ID and same modality)
        retrieved_hits = []
        for dist, h in zip(distances[0], raw_hits[0]):
            if gallery_pids[h] == q_pid and gallery_mods[h] == q_mod:
                continue
            retrieved_hits.append((h, dist))
            if len(retrieved_hits) == top_k:
                break

        # 4. Render Panel
        fig, axes = plt.subplots(1, top_k + 1, figsize=(19, 4.8), gridspec_kw={'wspace': 0.15})
        q_img, q_cmap = get_display_img(test_dataset, q_pid, q_mod)
        q_color = '#475569' if q_mod == 'SAR' else '#2563eb'
        axes[0].imshow(q_img, cmap=q_cmap)
        axes[0].set_title(f"QUERY [{q_mod.upper()}]\n{format_title_text(q_lbl)}", fontsize=8.5, color=q_color, fontweight='bold', pad=8)
        axes[0].axis('off')
        axes[0].add_patch(patches.Rectangle((0, 0), 1, 1, transform=axes[0].transAxes, fill=False, edgecolor=q_color, linewidth=4))

        for rank, (h, cos_score) in enumerate(retrieved_hits):
            h_pid, h_mod, m_lbl = gallery_pids[h], gallery_mods[h], gallery_labels[h]
            is_geographic_twin = (h_pid == q_pid)
            
            # Multi-Label Dice Overlap: 2 * |Q ∩ R| / (|Q| + |R|)
            overlap = q_lbl & m_lbl
            dice = 2 * len(overlap) / (len(q_lbl) + len(m_lbl)) if (len(q_lbl) + len(m_lbl)) > 0 else 0.0
            
            status_color = '#16a34a' if (dice > 0.0 or is_geographic_twin) else '#dc2626'
            twin_badge = " [TWIN]" if is_geographic_twin else ""

            r_img, r_cmap = get_display_img(test_dataset, h_pid, h_mod)
            axes[rank + 1].imshow(r_img, cmap=r_cmap)
            axes[rank + 1].set_title(
                f"Rank {rank+1} [{h_mod}]{twin_badge}\nDice: {dice:.2f} | Cos: {cos_score:.2f}\n{format_title_text(m_lbl)}",
                fontsize=8, color=status_color, fontweight='bold', pad=8
            )
            axes[rank + 1].axis('off')
            axes[rank + 1].add_patch(patches.Rectangle((0, 0), 1, 1, transform=axes[rank + 1].transAxes, fill=False, edgecolor=status_color, linewidth=3.5))

        plt.suptitle(
            f"Modality-Agnostic Test {test_idx + 1} | Query [{q_mod}] | End-to-End: {total_latency_ms:.1f} ms "
            f"(Encoder: {enc_latency_ms:.1f} ms, FAISS: {faiss_latency_ms:.3f} ms)",
            fontsize=11.5, fontweight='bold', y=0.98
        )
        plt.subplots_adjust(top=0.76, bottom=0.05)
        plt.show()

run_modality_agnostic_search(num_tests=4, top_k=5)